# circuit_runs

> Named multi-fly set-ups (Figs 3-5) for the circuit model.

In [ ]:
#| default_exp circuit_runs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from explRL.circuit import grid, goalStats, sinGoal
from explRL.circuit_train import (train_circuit, concatHistory, endTime, goalSafeDistance, heldValue,
                                  piScore, mapJumpProb, stablestCompass, probe)
from explRL.core import loadCircuitAgent
from explRL.envs import ZoneArena

In [ ]:
#| export
LENGTHS = dict(goal=1200.0, compass_asym=2400.0, compass_sym=4800.0, pretrain=2400.0,
               train=1200.0, probe=240.0, probeEvery=300.0)
SETUPS = ('goal', 'compass_asym', 'compass_sym', 'both')

def runLengths(lengths=None) -> dict:
    "Default run lengths in seconds (Q12), with named overrides; ValueError for unknown names."
    bad = set(lengths or {}) - set(LENGTHS)
    if bad:
        raise ValueError(f'unknown length name(s): {sorted(bad)}')
    return {**LENGTHS, **(lengths or {})}

def goalDistanceNow(agent, env) -> float:
    "Degrees from the agent's current goal (in arena coordinates) to the nearest safe-zone center."
    N = agent.params.nCirc
    g = grid(N)[(agent.a + agent.c) % N] - goalStats(agent.policy.wG)['thetaG']
    return env.safeDistance((g + np.pi) % (2 * np.pi) - np.pi)

def lastQuarterMean(history, values, start, T, tS) -> float:
    "Time average over the last quarter of [0, T] of values held between iterations (grid step <= 1 s)."
    times = np.arange(0.0, T, min(1.0, T / 100))
    held = heldValue(history, values, start, times, tS)
    return float(held[times >= 0.75 * T].mean())

In [ ]:
#| export
def _goal(params, seed, L):
    "Fig 3i: goal learning with a perfect compass (no compass learning, no jumps)."
    env = ZoneArena(N=params.nCirc, symmetric=True)
    ag = loadCircuitAgent(params=params, seed=seed, learnC=False, learnG=True, jumps=False)
    d0, s0 = goalDistanceNow(ag, env), goalStats(ag.policy.wG)['strength']
    h = train_circuit(ag, env, Ttot=L['goal'])
    return dict(history=h, wG=ag.policy.wG.copy(), wC=ag.policy.wC.copy(), distStart=d0, strStart=s0,
                distLastQ=lastQuarterMean(h, goalSafeDistance(h, env), d0, L['goal'], params.tS),
                strEnd=float(h.strength[-1]), PI=piScore(h))

def _compass(params, seed, L, sym):
    "Fig 4: compass learning with a fixed goal; asymmetric scene without jumps, symmetric with."
    N = params.nCirc
    env = ZoneArena(N=N, symmetric=sym)
    ag = loadCircuitAgent(params=params, seed=seed, wG0=sinGoal(0.4, grid(N)[8], N),
                          learnC=True, learnG=False, jumps=sym)
    before = probe(ag, env, T=L['probe'], seed=1000 + seed)
    h = train_circuit(ag, env, Ttot=L['compass_sym' if sym else 'compass_asym'])
    mirror = int(h.mirror[-1])
    mjp = mapJumpProb(ag.policy.wC, env, mirror)
    return dict(history=h, wG=ag.policy.wG.copy(), wC=ag.policy.wC.copy(), mirror=mirror, mapPJ=mjp,
                mapPJmean=float(mjp.mean()), probeBefore=before, trainJumpRate=float(h.jumped.mean()),
                probeAfter=probe(ag, env, T=L['probe'], seed=2000 + seed))

In [ ]:
#| export
def _both(params, seed, L):
    "Fig 5: asymmetric pre-training, goal reset to a danger center, then everything on in a symmetric scene."
    N, tS = params.nCirc, params.tS
    ag = loadCircuitAgent(params=params, seed=seed, learnC=True, learnG=False, jumps=False)
    train_circuit(ag, ZoneArena(N=N, symmetric=False), Ttot=L['pretrain'])
    ag.policy.wG = sinGoal(0.43, grid(N)[(ag.a + ag.c) % N], N)   # goal at arena 0 deg, strength 0.67
    ag.rl.learnG = ag.rl.jumps = True
    env = ZoneArena(N=N, symmetric=True)
    d0, s0 = goalDistanceNow(ag, env), goalStats(ag.policy.wG)['strength']
    marks = [float(m) for m in np.arange(L['probeEvery'], L['train'], L['probeEvery'])] + [L['train']]
    parts, t = [], 0.0
    pis = [probe(ag, env, T=L['probe'], seed=1000 + seed)['PI']]
    for k, mark in enumerate(marks, start=2):
        if mark > t:
            parts.append(train_circuit(ag, env, Ttot=mark - t, t0=t))
            t = endTime(parts[-1], tS)
        pis.append(probe(ag, env, T=L['probe'], seed=1000 * k + seed)['PI'])
    h = concatHistory(parts)
    stable = stablestCompass(ag.policy.wC, env, (ag.a + ag.c) % N)
    gap = (h.thetaG[-1] - stable + np.pi) % (2 * np.pi) - np.pi
    return dict(history=h, wG=ag.policy.wG.copy(), wC=ag.policy.wC.copy(), distStart=d0, strStart=s0,
                distEnd=goalDistanceNow(ag, env), strEnd=float(h.strength[-1]), probeTimes=[0.0] + marks,
                probePIs=pis, PI=piScore(h), stablest=stable, goalMinusStablest=float(np.rad2deg(gap)))

def runManyCircuit(setup, params, nRuns, seed=0, lengths=None) -> list:
    "Run nRuns flies (seeds seed + i) of one named set-up: 'goal', 'compass_asym', 'compass_sym' or 'both'."
    if setup not in SETUPS:
        raise ValueError(f'unknown setup {setup!r}')
    L = runLengths(lengths)
    run = dict(goal=_goal, both=_both, compass_asym=lambda p, s, L: _compass(p, s, L, False),
               compass_sym=lambda p, s, L: _compass(p, s, L, True))[setup]
    return [run(params, seed + i, L) for i in range(nRuns)]

In [ ]:
from fastcore.test import test_eq, test_close, test_fail
from explRL.params import Params
from explRL.circuit_train import CircuitHistory

p = Params()
test_eq(runLengths()['goal'], 1200.0)
test_eq(runLengths(dict(goal=5.0))['goal'], 5.0)
test_fail(lambda: runLengths(dict(nope=1.0)), contains='nope')
test_fail(lambda: runManyCircuit('nope', p, 1), contains='nope')

hand = CircuitHistory(t=np.array([0.0, 50.0]), a=np.array([0, 0]), c=np.array([0, 0]),
    direction=np.array([1, 1]), steps=np.array([1, 1]), jumped=np.array([False, False]),
    valence=np.array([1, 1]), fixDur=np.array([49.68, 49.68]), thetaG=np.array([0.0, 0.0]),
    strength=np.array([0.5, 0.9]), mirror=np.array([0, 0]))
test_close(lastQuarterMean(hand, hand.strength, 0.1, 100.0, p.tS), 0.5)   # first value held from 50 s to 100 s
test_close(lastQuarterMean(hand, hand.strength, 0.1, 40.0, p.tS), 0.1)    # nothing has ended yet

Short runs of every set-up return one dict per fly with the documented keys.

In [ ]:
short = dict(goal=10.0, compass_asym=10.0, compass_sym=10.0, pretrain=10.0, train=10.0, probe=5.0, probeEvery=2.5)
keys = dict(goal={'history', 'wG', 'wC', 'distStart', 'strStart', 'distLastQ', 'strEnd', 'PI'},
            compass_asym={'history', 'wG', 'wC', 'mirror', 'mapPJ', 'mapPJmean', 'probeBefore', 'trainJumpRate', 'probeAfter'},
            both={'history', 'wG', 'wC', 'distStart', 'strStart', 'distEnd', 'strEnd', 'probeTimes', 'probePIs',
                  'PI', 'stablest', 'goalMinusStablest'})
keys['compass_sym'] = keys['compass_asym']
for setup in SETUPS:
    runs = runManyCircuit(setup, p, 2, seed=10, lengths=short)
    test_eq(len(runs), 2)
    test_eq(set(runs[0]), keys[setup])
test_eq(runManyCircuit('compass_asym', p, 1, lengths=short)[0]['trainJumpRate'], 0.0)
rb = runManyCircuit('both', p, 1, seed=0, lengths=short)[0]
test_eq(rb['probeTimes'], [0.0, 2.5, 5.0, 7.5, 10.0])
test_eq(len(rb['probePIs']), 5)
test_close(rb['distStart'], 90.0)
assert np.all(np.diff(rb['history'].t) > 0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()